In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DWI_ZIP_PATH  = "/content/drive/MyDrive/datasets/DWI_NIFTI.zip"
MASK_ZIP_PATH = "/content/drive/MyDrive/datasets/MASK_NIFTI.zip"

In [3]:
import os, zipfile, glob

def safe_unzip(zip_path, target_dir):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(target_dir)

    # Recursively search .nii and .nii.gz BUT ignore the extracted_nii folder
    nii_files = []
    for root, dirs, files in os.walk(target_dir):

        # Skip duplicate folder
        if "extracted_nii" in root:
            continue

        for f in files:
            if f.endswith(".nii") or f.endswith(".nii.gz"):
                nii_files.append(os.path.join(root, f))

    return sorted(nii_files)

dwi_list = safe_unzip(DWI_ZIP_PATH, "/content/DWI_NIFTI_extracted")
mask_list = safe_unzip(MASK_ZIP_PATH, "/content/MASK_NIFTI_extracted")

print(f"# DWI volumes found: {len(dwi_list)}")
print(f"# MASK volumes found: {len(mask_list)}")
print("DWI examples:", dwi_list[:5])
print("MASK examples:", mask_list[:5])


# DWI volumes found: 250
# MASK volumes found: 250
DWI examples: ['/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0001_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0002_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0003_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0004_dwi.nii.gz', '/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0005_dwi.nii.gz']
MASK examples: ['/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0001_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0002_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0003_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0004_mask.nii.gz', '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0005_mask.nii.gz']


In [4]:
# Pairing rule
def pair_volumes(dwi_files, mask_files):
    def key(f):
        base = os.path.basename(f)
        base = os.path.splitext(base)[0]
        base = base.replace("_mask", "")
        base = base.replace("-mask", "")
        base = base.replace("_dwi", "")
        base = base.replace("-dwi", "")
        return base

    dwi_map = {key(f): f for f in dwi_files}
    mask_map = {key(f): f for f in mask_files}

    pairs = []
    for k, dwi_path in dwi_map.items():
        if k in mask_map:
            pairs.append((dwi_path, mask_map[k]))
    return pairs

pairs = pair_volumes(dwi_list, mask_list)
print("Paired volumes:", len(pairs))
pairs[:5]



Paired volumes: 250


[('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0001_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0001_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0002_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0002_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0003_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0003_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0004_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0004_mask.nii.gz'),
 ('/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0005_dwi.nii.gz',
  '/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0005_mask.nii.gz')]

In [5]:
!pip install monai[all] -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.

In [6]:
# CELL 1 — Imports & Setup
import os, glob, zipfile
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import nibabel as nib
import numpy as np
from sklearn.model_selection import train_test_split
from monai.networks.nets import resnet
from monai.losses import DiceCELoss

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device: cuda


In [7]:
# CELL 2 — Dataset Class & Transforms (Fixed)
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import numpy as np
from sklearn.model_selection import train_test_split

class DWIDataset(Dataset):
    def __init__(self, pairs, img_size=192, keep_empty_slices=False):
        self.img_size = img_size
        self.items = []
        self.keep_empty_slices = keep_empty_slices

        for dwi_path, mask_path in pairs:
            try:
                dwi_arr = nib.load(dwi_path).get_fdata(dtype=np.float32)
                mask_arr = nib.load(mask_path).get_fdata(dtype=np.float32)
            except:
                continue

            H, W, S = dwi_arr.shape
            mask_arr = mask_arr[:H, :W, :S]
            dwi_arr = dwi_arr[:H, :W, :S]

            for z in range(S):
                m_slice = mask_arr[:, :, z]
                if not keep_empty_slices and np.all(m_slice == 0):
                    continue
                self.items.append((dwi_path, mask_path, z))

        print(f"Prepared {len(self.items)} slices.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        dwi_path, mask_path, z = self.items[idx]

        img = nib.load(dwi_path).get_fdata(dtype=np.float32)[:, :, z]
        mask = nib.load(mask_path).get_fdata(dtype=np.float32)[:, :, z]

        # If slice has zero size, replace with zeros
        H, W = img.shape
        if H == 0 or W == 0:
            img = np.zeros((self.img_size, self.img_size), dtype=np.float32)
            mask = np.zeros((self.img_size, self.img_size), dtype=np.float32)

        # Normalize image
        mean, std = img.mean(), img.std()
        std = std if std > 0 else 1.0
        img = (img - mean) / std

        # Binarize mask
        mask = (mask > 0).astype(np.float32)

        # Convert to tensor and add channel dimension
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)   # [1, H, W]
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0) # [1, H, W]

        # Resize to fixed img_size
        img = F.interpolate(img.unsqueeze(0), size=(self.img_size, self.img_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        mask = F.interpolate(mask.unsqueeze(0), size=(self.img_size, self.img_size),
                             mode='nearest').squeeze(0)

        return img, mask


# Split datasets
dwi_train, dwi_val = train_test_split(pairs, test_size=0.2, random_state=42)
train_ds = DWIDataset(dwi_train, img_size=192)
val_ds   = DWIDataset(dwi_val, img_size=192)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=2)


Prepared 3957 slices.
Prepared 870 slices.


In [21]:
# ============================
#   SANITY CHECK FOR DWIDataset
# ============================

print("\n🔍 Running dataset sanity checks...\n")

datasets = [("train_ds", train_ds), ("val_ds", val_ds)]

for name, dataset in datasets:
    print(f"Checking {name} ...")
    print(f"  Total slices: {len(dataset)}")

    for i in range(min(10, len(dataset))):   # check first 10 samples
        try:
            img, mask = dataset[i]

            if img is None or mask is None:
                print(f"  ❗ None sample at index {i}")
                continue

            if img.shape != (1, 192, 192):
                print(f"  ❗ Bad IMG shape at index {i}: {img.shape}")

            if mask.shape != (1, 192, 192):
                print(f"  ❗ Bad MASK shape at index {i}: {mask.shape}")

            if torch.isnan(img).any():
                print(f"  ❗ NaNs in image at index {i}")

            if torch.isnan(mask).any():
                print(f"  ❗ NaNs in mask at index {i}")

        except Exception as e:
            print(f"  ❗ ERROR at index {i}: {e}")

print("\n✅ Dataset sanity check complete.\n")



🔍 Running dataset sanity checks...

Checking train_ds ...
  Total slices: 3957
Checking val_ds ...
  Total slices: 870

✅ Dataset sanity check complete.



In [35]:
print("Total DWI:", len(dwi_list))
print("Total MASK:", len(mask_list))

print("\nFirst 10 DWI paths:")
for p in dwi_list[:10]:
    print(p)

print("\nFirst 10 MASK paths:")
for p in mask_list[:10]:
    print(p)

print("\nPAIRS:")
for i, (d, m) in enumerate(pairs[:20]):
    print(i, "DWI:", d, " | MASK:", m)


Total DWI: 250
Total MASK: 250

First 10 DWI paths:
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0001_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0002_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0003_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0004_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0005_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0006_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0007_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0008_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0009_dwi.nii.gz
/content/DWI_NIFTI_extracted/DWI_NIFTI/sub-strokecase0010_dwi.nii.gz

First 10 MASK paths:
/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0001_mask.nii.gz
/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0002_mask.nii.gz
/content/MASK_NIFTI_extracted/MASK_NIFTI/sub-strokecase0003_mask.nii.gz
/content/MASK_NIFTI_

In [36]:
import nibabel as nib

for dwi, mask in pairs[:10]:
    img = nib.load(dwi).get_fdata()
    msk = nib.load(mask).get_fdata()
    print("DWI shape:", img.shape, "   MASK shape:", msk.shape)


DWI shape: (112, 112, 73)    MASK shape: (112, 112, 73)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)
DWI shape: (112, 112, 73)    MASK shape: (112, 112, 73)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)
DWI shape: (112, 112, 73)    MASK shape: (112, 112, 73)
DWI shape: (112, 112, 72)    MASK shape: (112, 112, 72)


In [10]:
# ####################################################################################
# ##############################################################################
# # Cell 3 — Define Model (Fully corrected for MONAI ResNet)

# import torch
# import torch.nn as nn
# from monai.networks.nets.resnet import ResNet, ResNetBottleneck

# class ResNet32Seg(nn.Module):
#     def __init__(self):
#         super().__init__()

#         # Correct MONAI ResNet initialization
#         self.backbone = ResNet(
#             block=ResNetBottleneck,
#             layers=(3, 3, 3, 3),                # ResNet-32 style
#             block_inplanes=[64, 128, 256, 512], # REQUIRED argument
#             spatial_dims=2,
#             n_input_channels=1,
#             num_classes=0                       # no classifier head
#         )

#         # Decoder with progressive upsampling

#         self.decoder = nn.Sequential(
#             nn.Conv2d(512, 256, 3, padding=1), nn.ReLU(),
#             nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 6→12

#             nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(),
#             nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 12→24

#             nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
#             nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 24→48

#             nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
#             nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 48→96

#             nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
#             nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),  # 96→192

#             nn.Conv2d(16, 1, 1)
#         )

#     def forward(self, x):
#         feats = self.backbone(x)
#         print("Backbone out:", feats.shape)  # should be [B, 512, 6, 6]
#         out = self.decoder(feats)
#         print("Decoder out:", out.shape)     # MUST be [B, 1, 192, 192]
#         return out

# model = ResNet32Seg().to(device)
# print("Model created successfully!")
# #############################################################

Initializing zero-element tensors is a no-op


Model created successfully!


In [37]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNet34Seg(nn.Module):
    def __init__(self):
        super().__init__()

        res = models.resnet34(weights=None)

        # --- MODIFY FIRST LAYER: accept 1-channel input (DWI is grayscale) ---
        res.conv1 = nn.Conv2d(
            1, 64,
            kernel_size=7, stride=2, padding=3, bias=False
        )

        # --- Backbone ---
        self.backbone = nn.Sequential(
            res.conv1,
            res.bn1,
            res.relu,
            res.maxpool,
            res.layer1,
            res.layer2,
            res.layer3,
            res.layer4,
        )
        # Output: [B, 512, 6, 6] for input 192×192

        # --- Decoder ---
        self.decoder = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),

            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        feats = self.backbone(x)
        #print("Backbone out:", feats.shape)

        out = self.decoder(feats)
        #print("Decoder out:", out.shape)

        return out

model = ResNet34Seg().to(device)
print("Model created successfully!")


Model created successfully!


In [38]:
# Cell 4 — Loss & Optimizer
import torch.nn.functional as F

# Dice Loss function
def dice_loss(preds, targets, smooth=1.0):
    preds = torch.sigmoid(preds)  # ensure in [0,1]
    preds_flat = preds.view(preds.size(0), -1)
    targets_flat = targets.view(targets.size(0), -1)
    intersection = (preds_flat * targets_flat).sum(dim=1)
    union = preds_flat.sum(dim=1) + targets_flat.sum(dim=1)
    dice = (2. * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()

# Combined BCE + Dice Loss
def bce_dice_loss(preds, targets):
    bce = F.binary_cross_entropy_with_logits(preds, targets)
    d_loss = dice_loss(preds, targets)
    return bce + d_loss

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [54]:
#Cell#5
def train_epoch(model, loader):
    model.train()
    epoch_loss = 0
    epoch_dice = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = bce_dice_loss(preds, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        # Dice score
        with torch.no_grad():
            dice = 1 - dice_loss(preds, masks)  # Dice metric
            epoch_dice += dice.item()
    return epoch_loss / len(loader), epoch_dice / len(loader)


def val_epoch(model, loader):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = bce_dice_loss(preds, masks)
            epoch_loss += loss.item()

            # Dice score
            dice = 1 - dice_loss(preds, masks)
            epoch_dice += dice.item()
    return epoch_loss / len(loader), epoch_dice / len(loader)


In [55]:
import os

EPOCHS = 20
CHECKPOINT_DIR = '/content/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_dice = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_dice = train_epoch(model, train_loader)
    val_loss, val_dice = val_epoch(model, val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")

    # Save best model
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pth'))
        print(f"Saved new best model with Dice: {best_dice:.4f}")

    # Save checkpoint every epoch
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pth'))


Epoch 01 | Train Loss: 0.5706 | Train Dice: 0.4624 | Val Loss: 0.6402 | Val Dice: 0.4028
Saved new best model with Dice: 0.4028
Epoch 02 | Train Loss: 0.5478 | Train Dice: 0.4837 | Val Loss: 0.5987 | Val Dice: 0.4383
Saved new best model with Dice: 0.4383
Epoch 03 | Train Loss: 0.5264 | Train Dice: 0.5028 | Val Loss: 0.5907 | Val Dice: 0.4393
Saved new best model with Dice: 0.4393
Epoch 04 | Train Loss: 0.5168 | Train Dice: 0.5120 | Val Loss: 0.6301 | Val Dice: 0.4200
Epoch 05 | Train Loss: 0.5009 | Train Dice: 0.5262 | Val Loss: 0.5893 | Val Dice: 0.4488
Saved new best model with Dice: 0.4488
Epoch 06 | Train Loss: 0.4986 | Train Dice: 0.5281 | Val Loss: 0.5872 | Val Dice: 0.4486
Epoch 07 | Train Loss: 0.4814 | Train Dice: 0.5445 | Val Loss: 0.5579 | Val Dice: 0.4719
Saved new best model with Dice: 0.4719
Epoch 08 | Train Loss: 0.4763 | Train Dice: 0.5495 | Val Loss: 0.5891 | Val Dice: 0.4504
Epoch 09 | Train Loss: 0.4654 | Train Dice: 0.5590 | Val Loss: 0.6006 | Val Dice: 0.4422
Epoc